# DINOv3 Strong-Lens Finder — Results

This notebook **auto-detects a trained checkpoint** at `runs/exp1/best.pt`:

* **Real mode** — if the checkpoint exists, it evaluates your fine-tuned DINOv3
  model and (re)writes the figures used in the README.
* **Demo mode** — otherwise it trains a quick **CPU baseline** (timm backbone, no
  DINOv3 download, no GPU) so the whole pipeline still runs end-to-end.

Either way it plots the ROC curve and the ranked-candidate grid. Run after
`pip install -e ".[sim,demo,notebook]"`.

In [ ]:
import os, sys
from pathlib import Path
# resolve paths to the repo root whether launched from the repo or from notebooks/
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
if (ROOT / 'src').is_dir():
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import matplotlib.pyplot as plt
from dino_lens_finder.config import Config
from dino_lens_finder.report import read_ranked, roc_figure, ranked_grid
print('working dir:', ROOT)

## 1. Evaluate the real checkpoint — or train a CPU baseline

In [ ]:
import torch
from dino_lens_finder.evaluation import evaluate_checkpoint

CKPT = 'runs/exp1/best.pt'
IS_REAL = os.path.exists(CKPT)

if IS_REAL:
    print('Real mode -> evaluating', CKPT)
    cfg = Config.from_dict(torch.load(CKPT, map_location='cpu').get('cfg'))
    RANKED = 'runs/exp1/ranked_candidates.csv'
    metrics = evaluate_checkpoint(cfg, CKPT, split='val', out=RANKED)
else:
    print('Demo mode -> training a quick CPU baseline (timm, no DINOv3/GPU)')
    DATA = 'data/demo'
    if not os.path.exists(f'{DATA}/index.csv'):
        try:
            from dino_lens_finder.simulation.lenstronomy_sim import build_dataset
            build_dataset(DATA, n_train=400, n_val=120, pos_frac=0.2, size=64)
        except ImportError:
            from dino_lens_finder.simulation.toy import build_dataset
            build_dataset(DATA, n_train=400, n_val=120, pos_frac=0.2, size=64)
    cfg = Config.from_dict({
        'backbone': {'source': 'timm', 'name': 'vit_small_patch16_224',
                     'img_size': 64, 'prefix_tokens': 1, 'pretrained': True},
        'finetune': {'mode': 'frozen'},
        'head': {'hidden_dim': 256, 'dropout': 0.2},
        'data': {'index': f'{DATA}/index.csv', 'img_size': 64, 'num_workers': 0,
                 'mean': [0.5, 0.5, 0.5], 'std': [0.5, 0.5, 0.5]},
        'train': {'epochs': 5, 'batch_size': 32, 'grad_accum': 1, 'amp': False,
                  'balance_sampler': True, 'out_dir': 'runs/demo'},
        'eval': {'fpr_target': 0.1, 'top_n': 16},
    })
    from dino_lens_finder.training.trainer import Trainer
    ckpt = Trainer(cfg).fit()
    RANKED = 'runs/demo/ranked.csv'
    metrics = evaluate_checkpoint(cfg, ckpt, split='val', out=RANKED)

# real-mode figures are written into assets/ (used by the README)
FIGDIR = 'assets' if IS_REAL else 'runs/demo'
os.makedirs(FIGDIR, exist_ok=True)
metrics

## 2. ROC curve
Accuracy is meaningless under extreme imbalance — read **AUC** and the operating
point **TPR@FPR**.

In [ ]:
rows = read_ranked(RANKED)
y = [r[2] for r in rows]
scores = [r[1] for r in rows]
roc_name = 'example_roc.png' if IS_REAL else 'roc.png'
fig = roc_figure(y, scores, cfg.eval.fpr_target)
fig.savefig(os.path.join(FIGDIR, roc_name), dpi=130, bbox_inches='tight')
print('saved', os.path.join(FIGDIR, roc_name))
plt.show()

## 3. Top-ranked candidates
Green border = true lens, red = false positive — a visual read of precision among
the candidates a human would inspect.

In [ ]:
grid_name = 'example_ranked_grid.png' if IS_REAL else 'top16.png'
fig = ranked_grid(rows, n=16, cols=4)
fig.savefig(os.path.join(FIGDIR, grid_name), dpi=130, bbox_inches='tight')
print('saved', os.path.join(FIGDIR, grid_name))
plt.show()

## 4. Notes & next steps

On simulated data the task is deliberately separable, so a near-perfect ROC just
validates the pipeline. The scientifically meaningful number is **sim→real**:

```bash
dino-lens make-bologna --image-dir <fits> --catalog <catalog.csv>
# point configs/default.yaml -> data.index at data/bologna/index.csv, then:
dino-lens eval --ckpt runs/exp1/best.pt --split val
```

Re-running this notebook in **real mode** regenerates `assets/example_roc.png` and
`assets/example_ranked_grid.png` (the README figures) from the current checkpoint.